# 208. Soft Prompt Tuning：怎样冻结基座并训练连续提示？

> **面试问题：怎样手写 soft prompt 前向/梯度更新，确保基座不变，并将 prompt id、形状、模板、租户和评测作为可发布制品管理？**

## 先给结论

这里的关键不是调用一个安全/训练/推理框架，而是定义输入、状态、不变量、失败分支和独立的判断 oracle。下方仅以受控小数据验证机制；真实服务仍需替换模型、密钥管理、访问控制、审计、红队评测和线上 SLO。

## 一手资料

- [Prompt Tuning](https://arxiv.org/abs/2104.08691)
- [Prefix-Tuning](https://arxiv.org/abs/2101.00190)
- [LoRA](https://arxiv.org/abs/2106.09685)

In [ ]:
notebook_contract = {"mode": "controlled-demo", "oracle": "assertions", "production": "security-and-versioning-required"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "controlled-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "versioning" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：soft prompt 只训练连续提示向量

Prompt tuning 冻结基座模型参数，学习若干虚拟 token 的 embedding。它和离散文字 prompt 不同，也不等于 LoRA 更新权重；训练、服务和缓存都必须带上 prompt artifact 版本。


In [ ]:
base_embeddings = {"good": (1.0, 0.0), "bad": (0.0, 1.0)}  # 执行本行的状态、计算或校验逻辑。
head = ((1.0, -1.0), (-1.0, 1.0))  # 执行本行的状态、计算或校验逻辑。
soft_prompt = [0.0, 0.0]  # 执行本行的状态、计算或校验逻辑。
assert base_embeddings["good"] == (1.0, 0.0)  # 执行本行的状态、计算或校验逻辑。
assert head[0] == (1.0, -1.0)  # 执行本行的状态、计算或校验逻辑。
assert soft_prompt == [0.0, 0.0]  # 执行本行的状态、计算或校验逻辑。


## 2. 前向：把虚拟 token 与冻结输入表征组合

教学模型以平均向量模拟 prompt + input 的上下文表征，再通过冻结线性 head 得到两类 logits。真实 Transformer 会把 soft prompt 放入 embedding 序列或各层 KV prefix，但“基座冻结、prompt 可训练”的参数边界相同。


In [ ]:
def dot(left, right):  # 执行本行的状态、计算或校验逻辑。
    return sum(a * b for a, b in zip(left, right))  # 执行本行的状态、计算或校验逻辑。
def forward(token, prompt):  # 执行本行的状态、计算或校验逻辑。
    context = tuple((base_embeddings[token][index] + prompt[index]) / 2 for index in range(2))  # 执行本行的状态、计算或校验逻辑。
    return tuple(dot(row, context) for row in head), context  # 执行本行的状态、计算或校验逻辑。
logits, context = forward("good", soft_prompt)  # 执行本行的状态、计算或校验逻辑。
assert context == (0.5, 0.0)  # 执行本行的状态、计算或校验逻辑。
assert logits == (0.5, -0.5)  # 执行本行的状态、计算或校验逻辑。
assert logits[0] > logits[1]  # 执行本行的状态、计算或校验逻辑。


## 3. 损失：交叉熵只对 soft prompt 求梯度

下面手写 softmax 和 `dL/dprompt`。head/base embedding 不更新；这让多个任务共享同一基座，代价是 prompt 容量和任务冲突仍需通过评测、路由或独立 prompt 管理。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
def softmax(values):  # 执行本行的状态、计算或校验逻辑。
    maximum = max(values)  # 执行本行的状态、计算或校验逻辑。
    exps = [math.exp(value - maximum) for value in values]  # 执行本行的状态、计算或校验逻辑。
    total = sum(exps)  # 执行本行的状态、计算或校验逻辑。
    return [value / total for value in exps]  # 执行本行的状态、计算或校验逻辑。
def loss_and_prompt_grad(token, prompt, label):  # 执行本行的状态、计算或校验逻辑。
    logits, _ = forward(token, prompt)  # 执行本行的状态、计算或校验逻辑。
    probs = softmax(logits)  # 执行本行的状态、计算或校验逻辑。
    gradient_logits = [probability - int(index == label) for index, probability in enumerate(probs)]  # 执行本行的状态、计算或校验逻辑。
    gradient = [sum(gradient_logits[row] * head[row][dimension] for row in range(2)) / 2 for dimension in range(2)]  # 执行本行的状态、计算或校验逻辑。
    return -math.log(probs[label]), gradient  # 执行本行的状态、计算或校验逻辑。
loss, gradient = loss_and_prompt_grad("bad", soft_prompt, 0)  # 执行本行的状态、计算或校验逻辑。
assert loss > 0  # 执行本行的状态、计算或校验逻辑。
assert len(gradient) == 2  # 执行本行的状态、计算或校验逻辑。
assert gradient[0] != gradient[1]  # 执行本行的状态、计算或校验逻辑。


## 4. 更新：只修改 prompt 参数并保留基座快照

SGD 更新需显式限定在 prompt 数组。真实训练还应保存 optimizer state、随机种子、数据/模板版本和训练步数；不要因代码便利把 base 权重放进 optimizer。


In [ ]:
def sgd_step(prompt, gradient, learning_rate):  # 执行本行的状态、计算或校验逻辑。
    return [value - learning_rate * delta for value, delta in zip(prompt, gradient)]  # 执行本行的状态、计算或校验逻辑。
updated_prompt = sgd_step(soft_prompt, gradient, 0.5)  # 执行本行的状态、计算或校验逻辑。
assert updated_prompt != soft_prompt  # 执行本行的状态、计算或校验逻辑。
assert base_embeddings == {"good": (1.0, 0.0), "bad": (0.0, 1.0)}  # 执行本行的状态、计算或校验逻辑。
assert head == ((1.0, -1.0), (-1.0, 1.0))  # 执行本行的状态、计算或校验逻辑。


## 5. 效果：更新后应以独立 loss/任务指标确认

一个单步更新不必总在任意样本上提高 loss，因此应对对应训练目标和独立验证集分开测。这里比较同一个目标的损失，体现“参数变化”不等于“成功适配”。


In [ ]:
new_loss, _ = loss_and_prompt_grad("bad", updated_prompt, 0)  # 执行本行的状态、计算或校验逻辑。
assert new_loss < loss  # 执行本行的状态、计算或校验逻辑。
assert new_loss > 0  # 执行本行的状态、计算或校验逻辑。
assert updated_prompt[0] != updated_prompt[1]  # 执行本行的状态、计算或校验逻辑。


## 6. 服务：prompt id 是请求/缓存/权限的一部分

多租户服务不能仅按 base checkpoint 做缓存。请求必须选择已发布的 prompt id，校验所属租户和 tokenizer/template；unknown 或未批准的 prompt 要拒绝，而不是静默使用默认配置。


In [ ]:
prompt_registry = {"tenant-a": {"sentiment-v1": tuple(updated_prompt)}}  # 执行本行的状态、计算或校验逻辑。
def load_prompt(tenant, prompt_id):  # 执行本行的状态、计算或校验逻辑。
    if prompt_id not in prompt_registry.get(tenant, {}):  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("prompt 不存在或不属于该租户")  # 执行本行的状态、计算或校验逻辑。
    return prompt_registry[tenant][prompt_id]  # 执行本行的状态、计算或校验逻辑。
assert load_prompt("tenant-a", "sentiment-v1") == tuple(updated_prompt)  # 执行本行的状态、计算或校验逻辑。
try:  # 执行本行的状态、计算或校验逻辑。
    load_prompt("tenant-b", "sentiment-v1")  # 执行本行的状态、计算或校验逻辑。
    assert False  # 执行本行的状态、计算或校验逻辑。
except ValueError:  # 执行本行的状态、计算或校验逻辑。
    assert True  # 执行本行的状态、计算或校验逻辑。


## 7. 失败分支：长度、模板或基座不匹配会改变语义

soft prompt 不是任意形状的向量文件。至少检查 hidden size、虚拟 token 数、base checkpoint、tokenizer 和 chat template；若不匹配，必须 fail closed 并要求重新训练或迁移。


In [ ]:
def compatible_prompt(artifact, runtime):  # 执行本行的状态、计算或校验逻辑。
    fields = ("base", "hidden_size", "tokens", "template")  # 执行本行的状态、计算或校验逻辑。
    return all(artifact[field] == runtime[field] for field in fields)  # 执行本行的状态、计算或校验逻辑。
prompt_artifact = {"base": "base-v1", "hidden_size": 2, "tokens": 1, "template": "chat-v1"}  # 执行本行的状态、计算或校验逻辑。
assert compatible_prompt(prompt_artifact, dict(prompt_artifact))  # 执行本行的状态、计算或校验逻辑。
assert not compatible_prompt(prompt_artifact, {**prompt_artifact, "hidden_size": 3})  # 执行本行的状态、计算或校验逻辑。
assert not compatible_prompt(prompt_artifact, {**prompt_artifact, "template": "chat-v2"})  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：基座、prompt、数据和评测绑定发布

一次可复放的软提示发布需要基座版本、prompt 参数 hash、训练数据快照、模板、指标和回滚目标。生产系统还应限制谁可上传/启用 prompt，并对更新做灰度与监控。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"base": "base-v1", "prompt": tuple(updated_prompt), "dataset": "sentiment-v1", "template": "chat-v1", "metric": new_loss}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["base"] == "base-v1"  # 执行本行的状态、计算或校验逻辑。
assert artifact["metric"] == new_loss  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

完整回答应覆盖目标、显式数据结构、核心规则、边界失败、评测指标和版本制品。受控断言只验证实现不变量，不能被解读为真实模型质量、攻击鲁棒性、隐私合规或线上成本结论。
